# OmniVoice — Server GPU trên Colab

Chạy tổng hợp giọng nói trên GPU Colab, máy ở nhà gọi lên qua API.

**Chỉ có client/server.** Không có giao diện, không có trình quản lý dự án —
đúng ba việc: nhiều worker INT4, hàng đợi, trả kết quả về client.

## Chạy theo thứ tự

1. Kiểm tra GPU
2. Build omnivoice.cpp (lần đầu 8–15 phút, cache lại được vào Drive)
3. Tải model INT4 (~660 MB)
4. Ghi mã Python
5. Khởi động server
6. Mở đường hầm ra ngoài → lấy URL + API key

Rồi ở máy mình:

```
python remote/client.py --url <URL> --key <KEY> ^
    --script scripts/kichban_pt.txt ^
    --ref output/refs3/FDown...-ref.wav --lang Portuguese ^
    --concurrency 4 -o output/remote/ket-qua.wav
```

## Số worker đặt bao nhiêu

Đo thật trên RTX 4000 Ada (`examples/bench_parallel.py`):

| worker | tăng tốc | VRAM | audio so với 1 luồng |
|---|---|---|---|
| 1 | 1.00x | 1443 MiB | mốc chuẩn |
| 2 | 1.80x | 2452 MiB | giống hệt từng byte |
| **4** | **2.60x** | 4802 MiB | giống hệt từng byte |
| 6 | 0.99x | 7140 MiB | giống hệt từng byte |
| 8 | 0.94x | 9440 MiB | giống hệt từng byte |

**4 là điểm tối ưu.** Quá 4 thì chậm đi chứ không nhanh thêm — GPU đã bão hoà.
Chia luồng **không đổi chất lượng**: băm SHA1 nội dung audio khớp 100% với bản
chạy tuần tự. T4 của Colab yếu hơn RTX 4000 Ada nên tốc độ tuyệt đối thấp hơn,
nhưng hình dạng đường cong thì giữ nguyên.

## Lưu ý

Phiên Colab tự ngắt sau vài giờ và mọi thứ trong `/content` mất theo. Bật
cache vào Drive ở ô số 2 thì lần sau khỏi build lại.


In [ ]:
# ── 1. Kiểm tra GPU ─────────────────────────────────────────────────────────
import shutil, subprocess, sys

if not shutil.which("nvidia-smi"):
    raise SystemExit(
        "Runtime này KHÔNG có GPU.\n"
        "Sửa: menu Runtime -> Change runtime type -> T4 GPU, rồi chạy lại từ đầu.\n"
        "Server này chỉ chạy CUDA, không có đường lui về CPU."
    )

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
cc = subprocess.run(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip().splitlines()[0]
CUDA_ARCH = cc.replace(".", "")
print(f"compute capability {cc} -> build riêng cho sm_{CUDA_ARCH} (nhanh hơn build đa kiến trúc)")


In [ ]:
# ── 2. Build omnivoice.cpp ──────────────────────────────────────────────────
# Lần đầu 8-15 phút. Bật USE_DRIVE_CACHE để lần sau khỏi build lại.
import os, subprocess, shutil
from pathlib import Path

USE_DRIVE_CACHE = False   # True -> mount Drive, cache thư mục build
DRIVE_CACHE = "/content/drive/MyDrive/omnivoice-build"

SRC = Path("/content/omnivoice.cpp")
BUILD = SRC / "build"
LIB = BUILD / "libomnivoice.so"

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")

def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, cwd=cwd)
    if r.returncode:
        raise SystemExit(f"lệnh thất bại: {cmd}")

if LIB.exists():
    print("đã có bản build sẵn.")
elif USE_DRIVE_CACHE and Path(DRIVE_CACHE, "libomnivoice.so").exists():
    print("khôi phục build từ Drive ...")
    SRC.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_CACHE, BUILD, dirs_exist_ok=True)
else:
    if not SRC.exists():
        sh("git clone --recurse-submodules --depth 1 "
           "https://github.com/ServeurpersoCom/omnivoice.cpp.git /content/omnivoice.cpp")
    sh(f"cmake -B build -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON "
       f"-DOMNIVOICE_SHARED=ON -DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH}", cwd=SRC)
    sh(f"cmake --build build -j$(nproc)", cwd=SRC)
    if USE_DRIVE_CACHE:
        Path(DRIVE_CACHE).mkdir(parents=True, exist_ok=True)
        for f in BUILD.glob("*.so"):
            shutil.copy2(f, Path(DRIVE_CACHE, f.name))
        print("đã lưu build vào Drive.")

assert LIB.exists(), "không thấy libomnivoice.so sau khi build"
os.environ["OMNIVOICE_LIB"] = str(BUILD)
print("thư viện:", LIB)
print(sorted(p.name for p in BUILD.glob("*.so")))


In [ ]:
# ── 3. Tải model INT4 (~660 MB) ─────────────────────────────────────────────
from pathlib import Path
from huggingface_hub import hf_hub_download

MODELS = Path("/content/models"); MODELS.mkdir(exist_ok=True)
for f in ["omnivoice-base-Q4_K_M.gguf", "omnivoice-tokenizer-Q8_0.gguf"]:
    if (MODELS / f).exists():
        print("[có sẵn]", f)
    else:
        print("[tải]", f)
        hf_hub_download("Serveurperso/OmniVoice-GGUF", f, local_dir=str(MODELS))
print(sorted(p.name for p in MODELS.glob("*.gguf")))


In [ ]:
# ── 4. Ghi mã Python ────────────────────────────────────────────────────────
# Nhúng sẵn trong notebook, không phải upload gì.
from pathlib import Path

APP = Path('/content/app'); APP.mkdir(exist_ok=True)
(APP / 'pyomnivoice').mkdir(exist_ok=True)

FILES = {}
FILES['pyomnivoice/__init__.py'] = '"""pyomnivoice - Python bindings for omnivoice.cpp.\n\nOmniVoice zero-shot TTS / voice cloning (646 languages, 24 kHz mono),\nrunning on CPU, CUDA, ROCm, Metal or Vulkan through GGML.\n"""\n\nfrom .core import (\n    FRAME_RATE,\n    PROFILES,\n    SAMPLE_RATE,\n    Audio,\n    OmniVoice,\n    OmniVoiceError,\n    Voice,\n    read_wav_24k,\n)\n\n__all__ = [\n    "OmniVoice",\n    "Voice",\n    "Audio",\n    "OmniVoiceError",\n    "read_wav_24k",\n    "PROFILES",\n    "SAMPLE_RATE",\n    "FRAME_RATE",\n]\n'
FILES['pyomnivoice/_ffi.py'] = '"""ctypes binding for the omnivoice.cpp public C ABI (ov_* symbols).\n\nMirrors src/omnivoice.h 1:1. Nothing here is meant to be called directly --\nuse pyomnivoice.OmniVoice.\n"""\n\nfrom __future__ import annotations\n\nimport ctypes\nimport os\nimport sys\nfrom ctypes import (\n    CFUNCTYPE,\n    POINTER,\n    Structure,\n    c_bool,\n    c_char_p,\n    c_float,\n    c_int,\n    c_int32,\n    c_uint64,\n    c_void_p,\n)\nfrom pathlib import Path\n\nOV_ABI_VERSION = 3\n\nOV_STATUS = {\n    0: "OK",\n    -1: "INVALID_PARAMS",\n    -2: "INSTRUCT_INVALID",\n    -3: "GENERATE_FAILED",\n    -4: "OOM",\n    -5: "CANCELLED",\n}\n\nLOG_LEVEL = {0: "DEBUG", 1: "INFO", 2: "WARN", 3: "ERROR"}\n\n\n# --------------------------------------------------------------------- structs\n\n\nclass ov_audio(Structure):\n    _fields_ = [\n        ("samples", POINTER(c_float)),\n        ("n_samples", c_int),\n        ("sample_rate", c_int),\n        ("channels", c_int),\n    ]\n\n\nclass ov_init_params(Structure):\n    _fields_ = [\n        ("abi_version", c_int),\n        ("model_path", c_char_p),\n        ("codec_path", c_char_p),\n        ("use_fa", c_bool),\n        ("clamp_fp16", c_bool),\n    ]\n\n\nov_cancel_cb = CFUNCTYPE(c_bool, c_void_p)\nov_audio_chunk_cb = CFUNCTYPE(c_bool, POINTER(c_float), c_int, c_void_p)\nov_log_cb = CFUNCTYPE(None, c_int, c_char_p, c_void_p)\n\n\nclass ov_tts_params(Structure):\n    _fields_ = [\n        ("abi_version", c_int),\n        # text / language / voice design\n        ("text", c_char_p),\n        ("lang", c_char_p),\n        ("instruct", c_char_p),\n        # duration + long form chunking\n        ("T_override", c_int),\n        ("chunk_duration_sec", c_float),\n        ("chunk_threshold_sec", c_float),\n        ("denoise", c_bool),\n        ("preprocess_prompt", c_bool),\n        # MaskGIT sampler\n        ("mg_num_step", c_int),\n        ("mg_guidance_scale", c_float),\n        ("mg_t_shift", c_float),\n        ("mg_layer_penalty_factor", c_float),\n        ("mg_position_temperature", c_float),\n        ("mg_class_temperature", c_float),\n        ("mg_seed", c_uint64),\n        # voice reference (tokens XOR raw pcm)\n        ("ref_audio_tokens", POINTER(c_int32)),\n        ("ref_T", c_int),\n        ("ref_audio_24k", POINTER(c_float)),\n        ("ref_n_samples", c_int),\n        ("ref_text", c_char_p),\n        ("dump_dir", c_char_p),\n        # callbacks\n        ("cancel", ov_cancel_cb),\n        ("cancel_user_data", c_void_p),\n        ("on_chunk", ov_audio_chunk_cb),\n        ("on_chunk_user_data", c_void_p),\n        # tail field, abi_version >= 3\n        ("postproc", c_bool),\n    ]\n\n\nclass ov_voice_ref(Structure):\n    _fields_ = [\n        ("ref_codes", POINTER(c_int32)),\n        ("ref_T", c_int),\n        ("num_codebooks", c_int),\n    ]\n\n\n# --------------------------------------------------------------------- loading\n\n_LIBNAME = {\n    "win32": "omnivoice.dll",\n    "darwin": "libomnivoice.dylib",\n}.get(sys.platform, "libomnivoice.so")\n\n_ROOT = Path(__file__).resolve().parent.parent\n\n# Where a build-win.cmd / buildcuda.sh run drops the shared library.\n_CANDIDATE_DIRS = [\n    _ROOT / "omnivoice.cpp" / "build-cuda",\n    _ROOT / "omnivoice.cpp" / "build-cuda" / "bin",\n    _ROOT / "omnivoice.cpp" / "build-cpu",\n    _ROOT / "omnivoice.cpp" / "build-cpu" / "bin",\n    _ROOT / "omnivoice.cpp" / "build" / "bin" / "Release",\n    _ROOT / "omnivoice.cpp" / "build" / "bin",\n    _ROOT / "omnivoice.cpp" / "build",\n    Path(__file__).resolve().parent / "lib",\n]\n\n\ndef find_library(explicit: str | os.PathLike | None = None) -> Path:\n    """Locate omnivoice.dll / libomnivoice.so.\n\n    Order: explicit argument, $OMNIVOICE_LIB, then the standard build dirs.\n    CUDA build wins over the CPU build when both are present.\n    """\n    if explicit:\n        p = Path(explicit)\n        if p.is_dir():\n            p = p / _LIBNAME\n        if not p.exists():\n            raise FileNotFoundError(f"omnivoice library not found at {p}")\n        return p\n\n    env = os.environ.get("OMNIVOICE_LIB")\n    if env:\n        return find_library(env)\n\n    for d in _CANDIDATE_DIRS:\n        p = d / _LIBNAME\n        if p.exists():\n            return p\n\n    raise FileNotFoundError(\n        f"{_LIBNAME} not found. Build it first:\\n"\n        f"    build-win.cmd cpu      (or: build-win.cmd cuda)\\n"\n        f"or point $OMNIVOICE_LIB at the directory holding {_LIBNAME}."\n    )\n\n\n_lib = None\n_lib_dir: Path | None = None\n\n\ndef load(explicit: str | os.PathLike | None = None):\n    """Load the shared library once and declare every prototype."""\n    global _lib, _lib_dir\n    if _lib is not None:\n        return _lib\n\n    path = find_library(explicit)\n    _lib_dir = path.parent\n\n    # ggml.dll / ggml-base.dll / ggml-cpu-*.dll sit next to it.\n    if sys.platform == "win32" and hasattr(os, "add_dll_directory"):\n        os.add_dll_directory(str(_lib_dir))\n\n    lib = ctypes.CDLL(str(path))\n\n    lib.ov_version.restype = c_char_p\n    lib.ov_version.argtypes = []\n\n    lib.ov_last_error.restype = c_char_p\n    lib.ov_last_error.argtypes = []\n\n    lib.ov_audio_free.restype = None\n    lib.ov_audio_free.argtypes = [POINTER(ov_audio)]\n\n    lib.ov_init_default_params.restype = None\n    lib.ov_init_default_params.argtypes = [POINTER(ov_init_params)]\n\n    lib.ov_init.restype = c_void_p\n    lib.ov_init.argtypes = [POINTER(ov_init_params)]\n\n    lib.ov_free.restype = None\n    lib.ov_free.argtypes = [c_void_p]\n\n    lib.ov_log_set.restype = None\n    lib.ov_log_set.argtypes = [ov_log_cb, c_void_p]\n\n    lib.ov_tts_default_params.restype = None\n    lib.ov_tts_default_params.argtypes = [POINTER(ov_tts_params)]\n\n    lib.ov_synthesize.restype = c_int\n    lib.ov_synthesize.argtypes = [c_void_p, POINTER(ov_tts_params), POINTER(ov_audio)]\n\n    lib.ov_duration_sec_to_tokens.restype = c_int\n    lib.ov_duration_sec_to_tokens.argtypes = [c_void_p, c_float]\n\n    lib.ov_num_codebooks.restype = c_int\n    lib.ov_num_codebooks.argtypes = [c_void_p]\n\n    lib.ov_extract_voice_ref.restype = c_int\n    lib.ov_extract_voice_ref.argtypes = [\n        c_void_p,\n        POINTER(c_float),\n        c_int,\n        POINTER(ov_voice_ref),\n    ]\n\n    lib.ov_voice_ref_free.restype = None\n    lib.ov_voice_ref_free.argtypes = [POINTER(ov_voice_ref)]\n\n    _lib = lib\n    return lib\n\n\ndef lib_dir() -> Path:\n    if _lib_dir is None:\n        raise RuntimeError("library not loaded yet")\n    return _lib_dir\n'
FILES['pyomnivoice/core.py'] = '"""High level Python API over omnivoice.cpp.\n\n    from pyomnivoice import OmniVoice\n\n    tts = OmniVoice(backend="cpu")                 # or "cuda" / "auto"\n    voice = tts.load_voice("ref.wav", "transcript of ref.wav")\n    audio = tts.say("Xin chao...", voice=voice, lang="Vietnamese")\n    audio.save("out.wav")\n"""\n\nfrom __future__ import annotations\n\nimport contextlib\nimport ctypes\nimport os\nimport time\nimport wave\nfrom ctypes import POINTER, byref, c_float, c_int32\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Callable, Iterable, Sequence\n\nimport numpy as np\n\nfrom . import _ffi\nfrom ._ffi import (\n    ov_audio,\n    ov_audio_chunk_cb,\n    ov_cancel_cb,\n    ov_init_params,\n    ov_log_cb,\n    ov_tts_params,\n    ov_voice_ref,\n)\n\nSAMPLE_RATE = 24_000\nFRAME_RATE = 25.0  # codec frames per second (hop 960 @ 24 kHz)\n\n_ROOT = Path(__file__).resolve().parent.parent\n_MODELS = _ROOT / "omnivoice.cpp" / "models"\n\n# (backbone, codec) presets, smallest first.\nPROFILES = {\n    # Nhỏ nhất, dành cho card 2 GB: codec cũng lượng tử hoá 4-bit.\n    "tiny": ("omnivoice-base-Q4_K_M.gguf", "omnivoice-tokenizer-Q4_K_M.gguf"),\n    "lite": ("omnivoice-base-Q4_K_M.gguf", "omnivoice-tokenizer-Q8_0.gguf"),\n    "balanced": ("omnivoice-base-Q8_0.gguf", "omnivoice-tokenizer-Q8_0.gguf"),\n    "quality": ("omnivoice-base-Q8_0.gguf", "omnivoice-tokenizer-F32.gguf"),\n    "reference": ("omnivoice-base-BF16.gguf", "omnivoice-tokenizer-F32.gguf"),\n}\n\n\nclass OmniVoiceError(RuntimeError):\n    pass\n\n\n# ------------------------------------------------------------------ audio i/o\n\n\n@dataclass\nclass Audio:\n    """Mono float32 PCM."""\n\n    samples: np.ndarray\n    sample_rate: int = SAMPLE_RATE\n\n    @property\n    def duration(self) -> float:\n        return len(self.samples) / self.sample_rate\n\n    def save(self, path: str | os.PathLike, bits: int = 16) -> Path:\n        path = Path(path)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        x = np.clip(self.samples, -1.0, 1.0)\n        if bits == 16:\n            data = (x * 32767.0).astype("<i2")\n        elif bits == 24:\n            i32 = (x * 8388607.0).astype("<i4")\n            data = i32.view(np.uint8).reshape(-1, 4)[:, :3].copy()\n        elif bits == 32:\n            data = x.astype("<f4")\n        else:\n            raise ValueError("bits must be 16, 24 or 32")\n        with wave.open(str(path), "wb") as w:\n            w.setnchannels(1)\n            w.setsampwidth(bits // 8)\n            w.setframerate(self.sample_rate)\n            w.writeframes(data.tobytes())\n        return path\n\n\ndef read_wav_24k(path: str | os.PathLike) -> np.ndarray:\n    """Decode any audio file to mono float32 @ 24 kHz.\n\n    Uses soundfile+soxr when installed, otherwise falls back to the stdlib\n    wave module plus linear interpolation so the package works on a bare\n    Python install.\n    """\n    path = str(path)\n    try:\n        import soundfile as sf\n\n        x, sr = sf.read(path, dtype="float32", always_2d=True)\n        x = x.mean(axis=1)\n    except Exception:\n        with contextlib.closing(wave.open(path, "rb")) as w:\n            sr = w.getframerate()\n            ch = w.getnchannels()\n            sw = w.getsampwidth()\n            raw = w.readframes(w.getnframes())\n        if sw == 2:\n            x = np.frombuffer(raw, dtype="<i2").astype(np.float32) / 32768.0\n        elif sw == 4:\n            x = np.frombuffer(raw, dtype="<i4").astype(np.float32) / 2147483648.0\n        elif sw == 1:\n            x = (np.frombuffer(raw, dtype=np.uint8).astype(np.float32) - 128.0) / 128.0\n        else:\n            raise OmniVoiceError(f"unsupported wav sample width: {sw}")\n        if ch > 1:\n            x = x.reshape(-1, ch).mean(axis=1)\n\n    if sr != SAMPLE_RATE:\n        try:\n            import soxr\n\n            x = soxr.resample(x, sr, SAMPLE_RATE, quality="VHQ")\n        except Exception:\n            n = int(round(len(x) * SAMPLE_RATE / sr))\n            x = np.interp(\n                np.linspace(0.0, len(x) - 1.0, n, dtype=np.float64),\n                np.arange(len(x), dtype=np.float64),\n                x.astype(np.float64),\n            )\n    return np.ascontiguousarray(x, dtype=np.float32)\n\n\n# ------------------------------------------------------------------- voice ref\n\n\n@dataclass\nclass Voice:\n    """A reusable voice reference: RVQ codes + the reference transcript.\n\n    Built once with OmniVoice.load_voice(), then reused for every sentence:\n    the codec encode (HuBERT + DAC + RVQ) is skipped on later calls.\n    """\n\n    codes: np.ndarray  # int32 [K, T]\n    text: str\n\n    @property\n    def n_frames(self) -> int:\n        return int(self.codes.shape[1])\n\n    def save(self, path: str | os.PathLike) -> Path:\n        path = Path(path)\n        np.savez(path, codes=self.codes, text=np.array(self.text))\n        return path\n\n    @staticmethod\n    def load(path: str | os.PathLike) -> "Voice":\n        z = np.load(str(path), allow_pickle=False)\n        return Voice(codes=z["codes"].astype(np.int32), text=str(z["text"]))\n\n\n# ----------------------------------------------------------------------- main\n\n\nclass OmniVoice:\n    """OmniVoice TTS: 646 languages, zero-shot voice cloning, 24 kHz mono.\n\n    Parameters\n    ----------\n    profile : "lite" | "balanced" | "quality" | "reference"\n        Model size preset. "lite" (Q4_K_M backbone + Q8_0 codec, ~660 MB\n        on disk) is the one for weak machines; "quality" matches the\n        INT4-backbone + FP32-Higgs-tokenizer pairing most launchers ship.\n    backend : "auto" | "cpu" | "cuda" | "vulkan" | explicit ggml device name\n        "auto" picks the best device present. "cpu" forces CPU even when a\n        GPU is available. Must be decided before the first synthesis.\n    """\n\n    def __init__(\n        self,\n        model: str | os.PathLike | None = None,\n        codec: str | os.PathLike | None = None,\n        *,\n        profile: str = "lite",\n        backend: str = "auto",\n        models_dir: str | os.PathLike | None = None,\n        lib: str | os.PathLike | None = None,\n        use_fa: bool = True,\n        clamp_fp16: bool = False,\n        verbose: bool = False,\n    ) -> None:\n        if profile not in PROFILES:\n            raise ValueError(f"profile must be one of {sorted(PROFILES)}")\n        mdir = Path(models_dir) if models_dir else _MODELS\n        dflt_model, dflt_codec = PROFILES[profile]\n        self.model_path = Path(model) if model else mdir / dflt_model\n        self.codec_path = Path(codec) if codec else mdir / dflt_codec\n        for p in (self.model_path, self.codec_path):\n            if not p.exists():\n                raise FileNotFoundError(\n                    f"{p} missing. Download the GGUFs with:\\n"\n                    f"    python -m pyomnivoice.download"\n                )\n\n        self._lib = _ffi.load(lib)\n        self.verbose = verbose\n        self._logs: list[str] = []\n        self._install_log_cb()\n\n        # ggml reads GGML_BACKEND at backend init time, inside ov_init.\n        dev = {"auto": None, "cpu": "CPU", "cuda": "CUDA0", "vulkan": "Vulkan0"}.get(\n            backend, backend\n        )\n        prev = os.environ.get("GGML_BACKEND")\n        if dev is None:\n            os.environ.pop("GGML_BACKEND", None)\n        else:\n            os.environ["GGML_BACKEND"] = dev\n\n        ip = ov_init_params()\n        self._lib.ov_init_default_params(byref(ip))\n        ip.model_path = str(self.model_path).encode("utf-8")\n        ip.codec_path = str(self.codec_path).encode("utf-8")\n        ip.use_fa = use_fa\n        ip.clamp_fp16 = clamp_fp16\n\n        t0 = time.perf_counter()\n        # ggml_backend_load_all() scans the cwd for ggml-cpu-*.dll variants.\n        cwd = os.getcwd()\n        try:\n            os.chdir(_ffi.lib_dir())\n            self._ctx = self._lib.ov_init(byref(ip))\n        finally:\n            os.chdir(cwd)\n            if prev is None:\n                os.environ.pop("GGML_BACKEND", None)\n            else:\n                os.environ["GGML_BACKEND"] = prev\n        self.load_time = time.perf_counter() - t0\n\n        if not self._ctx:\n            raise OmniVoiceError(\n                self._lib.ov_last_error().decode("utf-8", "replace")\n                or "ov_init failed"\n            )\n        self.backend = self._detect_backend()\n\n    # -- lifecycle ---------------------------------------------------------\n\n    def _install_log_cb(self) -> None:\n        def _cb(level: int, msg: bytes, _user):  # noqa: ANN001\n            text = msg.decode("utf-8", "replace").rstrip()\n            self._logs.append(text)\n            if self.verbose:\n                print(text, flush=True)\n\n        self._log_cb = ov_log_cb(_cb)  # keep a ref alive\n        self._lib.ov_log_set(self._log_cb, None)\n\n    def _detect_backend(self) -> str:\n        for line in self._logs:\n            if "backend:" in line:\n                return line.split("backend:")[1].split("(")[0].strip()\n        return "unknown"\n\n    def close(self) -> None:\n        if getattr(self, "_ctx", None):\n            self._lib.ov_free(self._ctx)\n            self._ctx = None\n\n    def __enter__(self) -> "OmniVoice":\n        return self\n\n    def __exit__(self, *exc) -> None:\n        self.close()\n\n    def __del__(self) -> None:\n        with contextlib.suppress(Exception):\n            self.close()\n\n    @property\n    def version(self) -> str:\n        return self._lib.ov_version().decode()\n\n    @property\n    def num_codebooks(self) -> int:\n        return int(self._lib.ov_num_codebooks(self._ctx))\n\n    def duration_to_frames(self, seconds: float) -> int:\n        return int(self._lib.ov_duration_sec_to_tokens(self._ctx, c_float(seconds)))\n\n    # -- voice cloning -----------------------------------------------------\n\n    def load_voice(self, ref_wav: str | os.PathLike, ref_text: str) -> Voice:\n        """Encode a reference WAV into reusable RVQ codes.\n\n        ref_wav: 3-10 s of clean speech, any sample rate/channel count.\n        ref_text: its exact transcript.\n        """\n        pcm = read_wav_24k(ref_wav)\n        out = ov_voice_ref()\n        rc = self._lib.ov_extract_voice_ref(\n            self._ctx,\n            pcm.ctypes.data_as(POINTER(c_float)),\n            len(pcm),\n            byref(out),\n        )\n        self._check(rc, "ov_extract_voice_ref")\n        n = out.num_codebooks * out.ref_T\n        codes = np.ctypeslib.as_array(out.ref_codes, shape=(n,)).copy()\n        codes = codes.reshape(out.num_codebooks, out.ref_T).astype(np.int32)\n        self._lib.ov_voice_ref_free(byref(out))\n        return Voice(codes=codes, text=ref_text)\n\n    # -- synthesis ---------------------------------------------------------\n\n    def say(\n        self,\n        text: str,\n        *,\n        voice: Voice | None = None,\n        ref_wav: str | os.PathLike | None = None,\n        ref_text: str | None = None,\n        instruct: str | None = None,\n        lang: str = "None",\n        duration: float | None = None,\n        steps: int = 32,\n        seed: int | None = None,\n        guidance_scale: float | None = None,\n        chunk_duration: float = 15.0,\n        chunk_threshold: float = 30.0,\n        denoise: bool = True,\n        preprocess_prompt: bool = True,\n        postproc: bool = True,\n        on_chunk: Callable[[np.ndarray], bool] | None = None,\n        cancel: Callable[[], bool] | None = None,\n    ) -> Audio:\n        """Synthesise `text`.\n\n        Voice selection, in priority order:\n          voice=Voice(...)            pre-encoded clone, cheapest\n          ref_wav= + ref_text=        clone straight from a WAV\n          instruct="female, young adult, moderate pitch"   voice design\n          nothing                     auto voice\n        """\n        p = ov_tts_params()\n        self._lib.ov_tts_default_params(byref(p))\n\n        # keep every buffer referenced until ov_synthesize returns\n        keep: list[object] = []\n\n        p.text = text.encode("utf-8")\n        p.lang = (lang or "None").encode("utf-8")\n        if instruct:\n            p.instruct = instruct.encode("utf-8")\n        p.chunk_duration_sec = chunk_duration\n        p.chunk_threshold_sec = chunk_threshold\n        p.denoise = denoise\n        p.preprocess_prompt = preprocess_prompt\n        p.postproc = postproc\n        p.mg_num_step = steps\n        if guidance_scale is not None:\n            p.mg_guidance_scale = guidance_scale\n        if seed is not None:\n            p.mg_seed = seed\n        if duration is not None:\n            p.T_override = self.duration_to_frames(duration)\n\n        if voice is not None:\n            codes = np.ascontiguousarray(voice.codes, dtype=np.int32)\n            keep.append(codes)\n            p.ref_audio_tokens = codes.ctypes.data_as(POINTER(c_int32))\n            p.ref_T = int(codes.shape[1])\n            p.ref_text = voice.text.encode("utf-8")\n        elif ref_wav is not None:\n            if not ref_text:\n                raise ValueError("ref_text is required together with ref_wav")\n            pcm = read_wav_24k(ref_wav)\n            keep.append(pcm)\n            p.ref_audio_24k = pcm.ctypes.data_as(POINTER(c_float))\n            p.ref_n_samples = len(pcm)\n            p.ref_text = ref_text.encode("utf-8")\n\n        chunks: list[np.ndarray] = []\n        if on_chunk is not None:\n            def _chunk_cb(ptr, n, _user):  # noqa: ANN001\n                buf = np.ctypeslib.as_array(ptr, shape=(n,)).copy()\n                chunks.append(buf)\n                return bool(on_chunk(buf))\n\n            cb = ov_audio_chunk_cb(_chunk_cb)\n            keep.append(cb)\n            p.on_chunk = cb\n\n        if cancel is not None:\n            def _cancel_cb(_user):  # noqa: ANN001\n                return bool(cancel())\n\n            ccb = ov_cancel_cb(_cancel_cb)\n            keep.append(ccb)\n            p.cancel = ccb\n\n        out = ov_audio()\n        t0 = time.perf_counter()\n        rc = self._lib.ov_synthesize(self._ctx, byref(p), byref(out))\n        wall = time.perf_counter() - t0\n        self._check(rc, "ov_synthesize")\n        del keep\n\n        if out.n_samples > 0:\n            pcm = np.ctypeslib.as_array(out.samples, shape=(out.n_samples,)).copy()\n            sr = out.sample_rate\n            self._lib.ov_audio_free(byref(out))\n        elif chunks:\n            pcm = np.concatenate(chunks)\n            sr = SAMPLE_RATE\n        else:\n            pcm = np.zeros(0, dtype=np.float32)\n            sr = SAMPLE_RATE\n\n        audio = Audio(samples=pcm, sample_rate=sr)\n        self.last_wall = wall\n        self.last_rtf = wall / audio.duration if audio.duration else float("nan")\n        return audio\n\n    def stream(\n        self, text: str, **kw\n    ) -> Iterable[np.ndarray]:\n        """Yield audio chunks as they are produced (generator wrapper)."""\n        import queue\n        import threading\n\n        q: "queue.Queue[np.ndarray | None]" = queue.Queue(maxsize=8)\n\n        def worker() -> None:\n            try:\n                self.say(text, on_chunk=lambda buf: (q.put(buf), True)[1], **kw)\n            finally:\n                q.put(None)\n\n        t = threading.Thread(target=worker, daemon=True)\n        t.start()\n        while True:\n            item = q.get()\n            if item is None:\n                break\n            yield item\n        t.join()\n\n    def say_many(\n        self, texts: Sequence[str], **kw\n    ) -> list[Audio]:\n        return [self.say(t, **kw) for t in texts]\n\n    def dub_srt(\n        self,\n        srt_path: str | os.PathLike,\n        *,\n        voice: Voice | None = None,\n        lang: str = "None",\n        steps: int = 32,\n        seed: int | None = None,\n        instruct: str | None = None,\n        progress: Callable[[int, int, "SrtCue"], None] | None = None,\n        **kw,\n    ) -> Audio:\n        """Dub an .srt onto an absolute timeline, ready to mux onto the video.\n\n        Each cue is forced to its exact slot length (T_override) with output\n        post-processing off, so nothing drifts; gaps between cues stay silent.\n        """\n        from .srt import assemble, read_srt\n\n        cues = read_srt(srt_path)\n        if not cues:\n            raise OmniVoiceError(f"no usable cues in {srt_path}")\n\n        segments = []\n        for i, cue in enumerate(cues):\n            if progress:\n                progress(i + 1, len(cues), cue)\n            audio = self.say(\n                cue.text,\n                voice=voice,\n                lang=lang,\n                instruct=instruct,\n                steps=steps,\n                seed=seed,\n                duration=cue.slot,\n                chunk_duration=0.0,  # single shot, the slot is the duration\n                postproc=False,\n                **kw,\n            )\n            segments.append((cue, audio.samples))\n\n        return Audio(assemble(segments, SAMPLE_RATE), SAMPLE_RATE)\n\n    # -- misc --------------------------------------------------------------\n\n    def _check(self, rc: int, what: str) -> None:\n        if rc != 0:\n            msg = self._lib.ov_last_error().decode("utf-8", "replace")\n            raise OmniVoiceError(f"{what} failed ({_ffi.OV_STATUS.get(rc, rc)}): {msg}")\n'
FILES['pyomnivoice/srt.py'] = '"""SubRip (.srt) parsing and timeline assembly.\n\nMirrors the --srt path of the omnivoice-tts CLI: every cue is synthesised\nsingle-shot with T_override set to its slot length and postproc off, so the\nraw decode lands at exactly the slot duration, then placed on an absolute\ntimeline that muxes straight onto the source video.\n"""\n\nfrom __future__ import annotations\n\nimport re\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\n\n_TIME = r"(\\d+):(\\d{1,2}):(\\d{1,2})[,.](\\d{1,3})"\n_ARROW = re.compile(rf"{_TIME}\\s*-->\\s*{_TIME}")\n\n\n@dataclass\nclass Cue:\n    index: int\n    t0: float\n    t1: float\n    text: str\n\n    @property\n    def slot(self) -> float:\n        return self.t1 - self.t0\n\n\ndef _stamp(h: str, m: str, s: str, ms: str) -> float:\n    frac = float(ms) / (10 ** len(ms)) if ms else 0.0\n    return int(h) * 3600 + int(m) * 60 + int(s) + frac\n\n\ndef parse_srt(text: str) -> list[Cue]:\n    """Tolerant of CRLF, a UTF-8 BOM, missing index lines, \'.\' or \',\' as the\n    millisecond separator, and multi-line cue text (joined with a space)."""\n    text = text.lstrip("\ufeff").replace("\\r\\n", "\\n").replace("\\r", "\\n")\n    cues: list[Cue] = []\n    for block in re.split(r"\\n\\s*\\n", text):\n        lines = [ln.strip() for ln in block.split("\\n") if ln.strip()]\n        if not lines:\n            continue\n        idx = 0\n        if lines[0].isdigit() and len(lines) > 1 and _ARROW.search(lines[1]):\n            idx = int(lines[0])\n            lines = lines[1:]\n        m = _ARROW.search(lines[0]) if lines else None\n        if not m:\n            continue\n        g = m.groups()\n        t0 = _stamp(*g[0:4])\n        t1 = _stamp(*g[4:8])\n        body = " ".join(lines[1:]).strip()\n        if body and t1 > t0:\n            cues.append(Cue(index=idx or len(cues) + 1, t0=t0, t1=t1, text=body))\n    cues.sort(key=lambda c: c.t0)\n    return cues\n\n\ndef read_srt(path: str | Path) -> list[Cue]:\n    raw = Path(path).read_text(encoding="utf-8-sig", errors="replace")\n    return parse_srt(raw)\n\n\ndef assemble(\n    segments: list[tuple[Cue, np.ndarray]],\n    sample_rate: int,\n    fade_ms: float = 5.0,\n) -> np.ndarray:\n    """Place each synthesised segment at its cue start on a zero timeline.\n\n    A segment is clipped at the next cue\'s start so an overlapping source\n    stamp never bleeds into the following line; a short raised-cosine fade on\n    both edges kills the click the raw (postproc-off) decode leaves behind.\n    """\n    if not segments:\n        return np.zeros(0, dtype=np.float32)\n\n    n_total = int(round(max(c.t1 for c, _ in segments) * sample_rate))\n    timeline = np.zeros(n_total, dtype=np.float32)\n    fade_n = int(sample_rate * fade_ms / 1000.0)\n\n    starts = [int(round(c.t0 * sample_rate)) for c, _ in segments]\n    for i, (cue, seg) in enumerate(segments):\n        off = starts[i]\n        limit = starts[i + 1] if i + 1 < len(starts) else n_total\n        limit = min(limit, n_total)\n        if off >= limit or seg.size == 0:\n            continue\n        n = min(len(seg), limit - off)\n        chunk = seg[:n].astype(np.float32, copy=True)\n        if fade_n > 0 and n > 2 * fade_n:\n            ramp = 0.5 - 0.5 * np.cos(np.pi * np.arange(fade_n) / fade_n)\n            chunk[:fade_n] *= ramp\n            chunk[-fade_n:] *= ramp[::-1]\n        timeline[off : off + n] = chunk\n    return timeline\n'
FILES['server.py'] = '"""Server tổng hợp giọng nói chạy trên GPU từ xa (Colab), nhiều worker + hàng đợi.\n\n    python remote/server.py --workers 4 --port 8770 --key BIMAT\n\nThiết kế đi thẳng từ số đo thật (`examples/bench_parallel.py`, RTX 4000 Ada):\n\n    worker   tăng tốc   VRAM       audio so với chạy 1 luồng\n      1        1.00x    1443 MiB   mốc chuẩn\n      2        1.80x    2452 MiB   giống hệt từng byte\n      4        2.60x    4802 MiB   giống hệt từng byte\n      6        0.99x    7140 MiB   giống hệt từng byte\n      8        0.94x    9440 MiB   giống hệt từng byte\n\nHai điều rút ra, đã đưa thẳng vào thiết kế:\n\n  - Chạy song song KHÔNG đổi chất lượng. Mỗi worker giữ một `ov_context`\n    riêng; băm SHA1 nội dung audio khớp 100% với bản chạy tuần tự cùng seed.\n    Nhờ vậy chia luồng thoải mái mà giọng không đổi.\n  - Quá 4 worker thì CHẬM ĐI chứ không nhanh thêm: GPU đã bão hoà, thêm\n    worker chỉ tốn VRAM. Mặc định 4, và server tự hạ xuống nếu VRAM không đủ.\n\nChỉ dùng thư viện chuẩn, không phải cài thêm gì trên Colab.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport base64\nimport io\nimport json\nimport os\nimport queue\nimport secrets\nimport subprocess\nimport sys\nimport threading\nimport time\nimport uuid\nimport wave\nfrom dataclasses import dataclass, field\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nfrom pathlib import Path\n\nimport numpy as np\n\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent))\n\nfrom pyomnivoice import Audio, OmniVoice, SAMPLE_RATE, Voice  # noqa: E402\n\nVRAM_PER_WORKER_MIB = 1250   # đo được: worker đầu 1443 MiB, mỗi worker sau ~1200\nMAX_USEFUL_WORKERS = 4       # quá mức này throughput đi xuống\n\n\n@dataclass\nclass Job:\n    text: str\n    voice_id: str | None\n    lang: str\n    steps: int\n    seed: int\n    instruct: str | None\n    done: threading.Event = field(default_factory=threading.Event)\n    audio: np.ndarray | None = None\n    error: str | None = None\n    wall: float = 0.0\n    queued_at: float = field(default_factory=time.perf_counter)\n    started_at: float = 0.0\n\n\nclass Pool:\n    """N engine độc lập, một hàng đợi chung, phục vụ theo thứ tự đến trước."""\n\n    def __init__(self, n: int, profile: str, models_dir: Path | None) -> None:\n        self.q: "queue.Queue[Job | None]" = queue.Queue()\n        self.voices: dict[str, tuple[str, Voice]] = {}\n        self.lock = threading.Lock()\n        self.busy = 0\n        self.served = 0\n        self.failed = 0\n        self.audio_sec = 0.0\n        self.synth_sec = 0.0\n        self.started = time.time()\n\n        print(f"[server] nap {n} engine, profile {profile} ...", flush=True)\n        t0 = time.perf_counter()\n        self.engines = [\n            OmniVoice(profile=profile, backend="cuda", models_dir=models_dir)\n            for _ in range(n)\n        ]\n        self.backend = self.engines[0].backend\n        print(f"[server] xong sau {time.perf_counter() - t0:.1f}s, backend {self.backend}",\n              flush=True)\n        if "CUDA" not in (self.backend or "").upper():\n            raise SystemExit("Backend khong phai CUDA. Server nay chi chay GPU.")\n\n        self.threads = [threading.Thread(target=self._worker, args=(i,), daemon=True)\n                        for i in range(n)]\n        for t in self.threads:\n            t.start()\n\n    def add_voice(self, name: str, pcm: np.ndarray, text: str) -> str:\n        """Mã hoá giọng mẫu MỘT lần trên engine 0.\n\n        Mã RVQ chỉ phụ thuộc codec nên mọi worker dùng chung được. Voice là\n        mảng numpy chỉ đọc, chia sẻ giữa các luồng an toàn.\n        """\n        vid = uuid.uuid4().hex[:12]\n        tmp = Path(os.environ.get("TMPDIR", "/tmp")) / f"ref-{vid}.wav"\n        tmp.parent.mkdir(parents=True, exist_ok=True)\n        Audio(pcm).save(tmp)\n        try:\n            voice = self.engines[0].load_voice(tmp, text)\n        finally:\n            tmp.unlink(missing_ok=True)\n        with self.lock:\n            self.voices[vid] = (name, voice)\n        print(f"[server] giong \'{name}\' -> {vid} ({voice.n_frames} frame)", flush=True)\n        return vid\n\n    def submit(self, job: Job) -> None:\n        self.q.put(job)\n\n    def _worker(self, wid: int) -> None:\n        eng = self.engines[wid]\n        while True:\n            job = self.q.get()\n            if job is None:\n                return\n            with self.lock:\n                self.busy += 1\n            job.started_at = time.perf_counter()\n            try:\n                voice = None\n                if job.voice_id:\n                    with self.lock:\n                        entry = self.voices.get(job.voice_id)\n                    if entry is None:\n                        raise KeyError(f"voice_id khong ton tai: {job.voice_id}")\n                    voice = entry[1]\n                a = eng.say(job.text, voice=voice, instruct=job.instruct,\n                            lang=job.lang, steps=job.steps, seed=job.seed)\n                job.audio = a.samples\n                job.wall = eng.last_wall\n                with self.lock:\n                    self.served += 1\n                    self.audio_sec += a.duration\n                    self.synth_sec += eng.last_wall\n            except Exception as e:  # noqa: BLE001\n                job.error = f"{type(e).__name__}: {e}"\n                with self.lock:\n                    self.failed += 1\n            finally:\n                with self.lock:\n                    self.busy -= 1\n                job.done.set()\n\n    def stats(self) -> dict:\n        with self.lock:\n            return {\n                "workers": len(self.engines),\n                "busy": self.busy,\n                "queued": self.q.qsize(),\n                "served": self.served,\n                "failed": self.failed,\n                "audio_sec": round(self.audio_sec, 1),\n                "synth_sec": round(self.synth_sec, 1),\n                "xrt_aggregate": (round(self.audio_sec / self.synth_sec, 2)\n                                  if self.synth_sec else None),\n                "uptime_sec": round(time.time() - self.started),\n                "voices": {k: v[0] for k, v in self.voices.items()},\n                "backend": self.backend,\n            }\n\n\ndef gpu_info() -> dict:\n    try:\n        out = subprocess.run(\n            ["nvidia-smi",\n             "--query-gpu=name,memory.total,memory.free,compute_cap",\n             "--format=csv,noheader,nounits"],\n            capture_output=True, text=True, timeout=10).stdout.splitlines()[0]\n        n, tot, free, cc = [x.strip() for x in out.split(",")]\n        return {"name": n, "vram_total_mib": int(tot),\n                "vram_free_mib": int(free), "compute_cap": cc}\n    except Exception as e:  # noqa: BLE001\n        return {"error": str(e)}\n\n\ndef wav_bytes(pcm: np.ndarray, sr: int = SAMPLE_RATE) -> bytes:\n    buf = io.BytesIO()\n    with wave.open(buf, "wb") as w:\n        w.setnchannels(1)\n        w.setsampwidth(2)\n        w.setframerate(sr)\n        w.writeframes((np.clip(pcm, -1, 1) * 32767).astype("<i2").tobytes())\n    return buf.getvalue()\n\n\ndef read_wav_bytes(b: bytes) -> np.ndarray:\n    with wave.open(io.BytesIO(b), "rb") as w:\n        sr, ch, sw = w.getframerate(), w.getnchannels(), w.getsampwidth()\n        raw = w.readframes(w.getnframes())\n    if sw != 2:\n        raise ValueError("chi nhan WAV 16-bit")\n    x = np.frombuffer(raw, "<i2").astype(np.float32) / 32768.0\n    if ch > 1:\n        x = x.reshape(-1, ch).mean(axis=1)\n    if sr != SAMPLE_RATE:\n        n = int(round(len(x) * SAMPLE_RATE / sr))\n        x = np.interp(np.linspace(0, len(x) - 1, n), np.arange(len(x)), x).astype(np.float32)\n    return np.ascontiguousarray(x, dtype=np.float32)\n\n\nPOOL: Pool | None = None\nAPI_KEY = ""\n\n\nclass Handler(BaseHTTPRequestHandler):\n    protocol_version = "HTTP/1.1"\n    server_version = "omnivoice-remote/1"\n\n    def log_message(self, fmt, *args):\n        code = str(args[1]) if len(args) > 1 else ""\n        if not code.startswith("2"):\n            sys.stderr.write("[http] " + (fmt % args) + "\\n")\n\n    def _send(self, code: int, body: bytes, ctype: str = "application/json") -> None:\n        self.send_response(code)\n        self.send_header("Content-Type", ctype)\n        self.send_header("Content-Length", str(len(body)))\n        self.send_header("Access-Control-Allow-Origin", "*")\n        self.send_header("Access-Control-Allow-Headers", "Content-Type, X-API-Key")\n        self.end_headers()\n        self.wfile.write(body)\n\n    def _json(self, code: int, obj) -> None:\n        self._send(code, json.dumps(obj, ensure_ascii=False).encode("utf-8"))\n\n    def _auth(self) -> bool:\n        if not API_KEY or self.headers.get("X-API-Key") == API_KEY:\n            return True\n        self._json(401, {"error": "thieu hoac sai X-API-Key"})\n        return False\n\n    def _body(self) -> dict:\n        n = int(self.headers.get("Content-Length") or 0)\n        return json.loads(self.rfile.read(n) or b"{}")\n\n    def do_OPTIONS(self):  # noqa: N802\n        self._send(204, b"", "text/plain")\n\n    def do_GET(self):  # noqa: N802\n        assert POOL is not None\n        if self.path.startswith("/health"):\n            self._json(200, {"ok": True, "gpu": gpu_info(), **POOL.stats()})\n        elif self.path.startswith("/voices"):\n            if self._auth():\n                self._json(200, {"voices": POOL.stats()["voices"]})\n        else:\n            self._json(404, {"error": "khong co route nay"})\n\n    def do_POST(self):  # noqa: N802\n        if not self._auth():\n            return\n        assert POOL is not None\n        try:\n            if self.path.startswith("/voice"):\n                d = self._body()\n                pcm = read_wav_bytes(base64.b64decode(d["wav_b64"]))\n                vid = POOL.add_voice(d.get("name", "voice"), pcm, d["text"])\n                self._json(200, {"voice_id": vid,\n                                 "seconds": round(len(pcm) / SAMPLE_RATE, 2)})\n\n            elif self.path.startswith("/tts"):\n                d = self._body()\n                if not d.get("text"):\n                    self._json(400, {"error": "thieu \'text\'"})\n                    return\n                job = Job(text=d["text"], voice_id=d.get("voice_id"),\n                          lang=d.get("lang", "None"), steps=int(d.get("steps", 16)),\n                          seed=int(d.get("seed", 42)), instruct=d.get("instruct"))\n                POOL.submit(job)\n                if not job.done.wait(timeout=float(d.get("timeout", 900))):\n                    self._json(504, {"error": "qua han cho"})\n                    return\n                if job.error:\n                    self._json(500, {"error": job.error})\n                    return\n                b = wav_bytes(job.audio)\n                self.send_response(200)\n                self.send_header("Content-Type", "audio/wav")\n                self.send_header("Content-Length", str(len(b)))\n                self.send_header("X-Synth-Seconds", f"{job.wall:.3f}")\n                self.send_header("X-Queue-Seconds", f"{job.started_at - job.queued_at:.3f}")\n                self.send_header("X-Audio-Seconds", f"{len(job.audio) / SAMPLE_RATE:.3f}")\n                self.send_header("Access-Control-Allow-Origin", "*")\n                self.end_headers()\n                self.wfile.write(b)\n            else:\n                self._json(404, {"error": "khong co route nay"})\n        except Exception as e:  # noqa: BLE001\n            self._json(500, {"error": f"{type(e).__name__}: {e}"})\n\n\ndef main() -> None:\n    global POOL, API_KEY\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--workers", type=int, default=4)\n    ap.add_argument("--port", type=int, default=8770)\n    ap.add_argument("--host", default="0.0.0.0")\n    ap.add_argument("--profile", default="lite", help="lite = INT4")\n    ap.add_argument("--models-dir", default=None)\n    ap.add_argument("--key", default=None, help="API key; bo trong se tu sinh")\n    ap.add_argument("--allow-oversubscribe", action="store_true",\n                    help="cho phep vuot 4 worker du do duoc la cham hon")\n    args = ap.parse_args()\n\n    info = gpu_info()\n    print(f"[server] GPU: {info}", flush=True)\n\n    n = args.workers\n    if not args.allow_oversubscribe and n > MAX_USEFUL_WORKERS:\n        print(f"[server] {n} worker vuot muc huu ich, ha ve {MAX_USEFUL_WORKERS}. "\n              f"Do duoc: 6 worker = 0.99x, 8 worker = 0.94x so voi 4.", flush=True)\n        n = MAX_USEFUL_WORKERS\n    free = info.get("vram_free_mib")\n    if free:\n        fit = max(1, (free - 400) // VRAM_PER_WORKER_MIB)\n        if n > fit:\n            print(f"[server] VRAM trong {free} MiB chi du {fit} worker, "\n                  f"ha tu {n} xuong {fit}.", flush=True)\n            n = fit\n\n    API_KEY = args.key or secrets.token_urlsafe(12)\n    POOL = Pool(n, args.profile, Path(args.models_dir) if args.models_dir else None)\n\n    srv = ThreadingHTTPServer((args.host, args.port), Handler)\n    srv.daemon_threads = True\n    print("=" * 66, flush=True)\n    print(f"  Server san sang   http://{args.host}:{args.port}", flush=True)\n    print(f"  Worker            {n}", flush=True)\n    print(f"  API key           {API_KEY}", flush=True)\n    print("=" * 66, flush=True)\n    try:\n        srv.serve_forever()\n    except KeyboardInterrupt:\n        print("\\n[server] dung.", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'

for name, body in FILES.items():
    p = APP / name
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(body, encoding='utf-8')
    print(f'{len(body):7d} ký tự  {name}')

In [ ]:
# ── 5. Khởi động server ─────────────────────────────────────────────────────
import json, os, secrets, subprocess, time, urllib.request

PORT = 8770
WORKERS = 4          # 4 là điểm tối ưu đo được; server tự hạ nếu VRAM không đủ
API_KEY = secrets.token_urlsafe(12)
LOG = "/content/server.log"

subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
time.sleep(1)

env = dict(os.environ, OMNIVOICE_LIB="/content/omnivoice.cpp/build",
           PYTHONUNBUFFERED="1")
with open(LOG, "wb") as f:
    subprocess.Popen(
        ["python", "/content/app/server.py", "--workers", str(WORKERS),
         "--port", str(PORT), "--key", API_KEY,
         "--models-dir", "/content/models", "--profile", "lite"],
        stdout=f, stderr=subprocess.STDOUT, env=env, cwd="/content/app")

for i in range(180):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=3) as r:
            h = json.load(r)
        print(json.dumps(h, ensure_ascii=False, indent=2))
        print(f"\nSERVER SẴN SÀNG — {h['workers']} worker")
        break
    except Exception:
        time.sleep(2)
else:
    print(open(LOG, encoding="utf-8", errors="replace").read()[-4000:])
    raise SystemExit("server không lên. Xem log ở trên.")


In [ ]:
# ── 6. Mở đường hầm ra ngoài ────────────────────────────────────────────────
# cloudflared: không cần đăng ký tài khoản, URL dùng ngay.
import re, subprocess, time
from pathlib import Path

if not Path("/content/cloudflared").exists():
    subprocess.run(
        "wget -q -O /content/cloudflared "
        "https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64 && chmod +x /content/cloudflared", shell=True, check=True)

subprocess.run("pkill -f cloudflared || true", shell=True)
subprocess.Popen(
    f"/content/cloudflared tunnel --url http://127.0.0.1:{PORT} --no-autoupdate "
    f"> /content/cloudflared.log 2>&1", shell=True)

URL = None
for _ in range(40):
    time.sleep(2)
    log = Path("/content/cloudflared.log").read_text(errors="replace")
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
    if m:
        URL = m.group(0)
        break

if not URL:
    print(Path("/content/cloudflared.log").read_text(errors="replace")[-3000:])
    raise SystemExit("không lấy được URL đường hầm")

print("=" * 70)
print("  CHÉP HAI DÒNG NÀY VỀ MÁY MÌNH")
print("=" * 70)
print(f"  URL      {URL}")
print(f"  API key  {API_KEY}")
print("=" * 70)
print()
print("Lệnh chạy ở máy mình:")
print()
print(f"  python remote/client.py --url {URL} --key {API_KEY} \\")
print( "      --script scripts/kichban_pt.txt \\")
print( "      --ref output/refs3/FDown.vn_Tai_video_Facebook_MP3_9995-ref.wav \\")
print( "      --lang Portuguese --concurrency 4 -o output/remote/ket-qua.wav")


In [ ]:
# ── 7. Tự kiểm tra (tuỳ chọn) ───────────────────────────────────────────────
# Gửi thử một câu qua chính đường hầm, đo độ trễ khứ hồi.
import json, time, urllib.request

t0 = time.perf_counter()
req = urllib.request.Request(URL + "/tts", method="POST",
    data=json.dumps({"text": "Xin chào, đây là bài kiểm tra kết nối.",
                     "lang": "Vietnamese", "steps": 16}).encode(),
    headers={"Content-Type": "application/json", "X-API-Key": API_KEY})
with urllib.request.urlopen(req, timeout=300) as r:
    wav = r.read()
    synth = float(r.headers.get("X-Synth-Seconds", 0))
    audio = float(r.headers.get("X-Audio-Seconds", 0))
rtt = time.perf_counter() - t0

open("/content/test.wav", "wb").write(wav)
print(f"audio {audio:.2f}s | GPU tính {synth:.2f}s | khứ hồi qua đường hầm {rtt:.2f}s")
print(f"phần mạng chiếm {rtt - synth:.2f}s")
from IPython.display import Audio, display
display(Audio("/content/test.wav"))
